In [1]:
import numpy as np
from scipy import stats
import os 
import math
import pandas as pd

os.chdir('../../')
from metric import calculate_correlation_with_groundtruth, calculate_precision_racall
from binlib import output_bin

ref_genome_folder = 'Data/ref_genome/hg19'
exclude_chr = ['chrM', 'chrY', 'chrMT']
resolution = 50000

# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)

# back to current dir
os.chdir('Exps/exp3_pool_residual')

chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


In [4]:
bin_dir = '../../CNV/K562_HIC_CNV/HiNTcnv_OUTPUT_K562/segmentation/b50000'

res_dict = dict()
for key in chr_bin_number.keys():
    res_dict[key] = np.zeros(chr_bin_number[key])

keys = list(chr_bin_number.keys())
key_num = 0
for file in os.listdir(bin_dir):
    with open(f'{bin_dir}/{file}', 'r') as f:
        for line in f.readlines()[1:]:
            start_, end_, obs, exp ,var = line.strip().split()
            residual = float(obs) - float(exp)
            bin_ = math.floor(int(start_)/resolution)
            res_dict[keys[key_num]][bin_] = residual
    key_num = key_num +1

In [2]:
win_size_list = [3,5,11,21,41]

### operate averge pooling in defferent window size

In [17]:

for win_size in win_size_list:
    pool_res_dict = dict()
    for key in chr_bin_number.keys():
        pool_res_dict[key] = np.zeros(chr_bin_number[key])


    for chr_ in res_dict.keys():
        for i in range(len(res_dict[chr_])):
            start_ = i - math.floor(win_size/2) if i - math.floor(win_size/2) >=0 else 0 
            end_ = i + math.floor(win_size/2) if i + math.floor(win_size/2) <= chr_bin_number[chr_] else chr_bin_number[chr_]
            win_ = res_dict[chr_][start_:end_+1]
            pool_res_dict[chr_][i] = np.mean(win_)

    #output .bw for visualization
    output_path = f'hic_residual_pool_win{win_size}.bw'
    with open(output_path, 'w+') as f:
        for key in pool_res_dict.keys():
            for bin_, residual in enumerate(pool_res_dict[key]):
                bin_start = bin_ * 50000
                bin_end = (bin_+1) * 50000 if (bin_+1) * 50000 <= chr_length[key] else chr_length[key]
                f.write(f'{key}\t{bin_start}\t{bin_end}\t{residual}\n')

    ### output for BICseq-seg
    output_bin(pool_res_dict, f'win{win_size}', f'pool_win{win_size}')

### need to run BICseq-seg manually

### metrics

In [5]:
for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_correlation_with_groundtruth(f'win{win_size}/pool_win{win_size}_seg.txt', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Pearson r :  0.876
Spearman r :  0.884


win size : 5
Pearson r :  0.867
Spearman r :  0.864


win size : 11
Pearson r :  0.818
Spearman r :  0.839


win size : 21
Pearson r :  0.745
Spearman r :  0.833


win size : 41
Pearson r :  0.633
Spearman r :  0.831




In [3]:

for win_size in win_size_list:
    print(f'win size : {win_size}')
    calculate_precision_racall(f'win{win_size}/pool_win{win_size}_seg.txt', groundtruth_file = "../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt")
    print('\n')

win size : 3
Number of ground truth : 74
Number of target : 34
Number of hits : 27
precision : 0.794
recall : 0.365
average hit fraction : 0.9


win size : 5
Number of ground truth : 74
Number of target : 36
Number of hits : 29
precision : 0.806
recall : 0.392
average hit fraction : 0.892


win size : 11
Number of ground truth : 74
Number of target : 39
Number of hits : 26
precision : 0.667
recall : 0.351
average hit fraction : 0.878


win size : 21
Number of ground truth : 74
Number of target : 40
Number of hits : 26
precision : 0.65
recall : 0.351
average hit fraction : 0.844


win size : 41
Number of ground truth : 74
Number of target : 46
Number of hits : 26
precision : 0.565
recall : 0.351
average hit fraction : 0.818


